# Calibration du scaling — moteur custom vs ×N naïf

**Objectif.** Visualiser où le moteur de scaling déterministe (`app/scaling/`)
**diverge** d'une multiplication naïve `×N` (comportement Cooklang binaire, D6)
et où il l'**égale**. C'est l'outil de la boucle de calibration **F2** et la
réponse au risque **R3** (« coefficients faux »).

- **×N naïf** = `qty × k` pour *toutes* les quantités, indistinctement.
- **moteur** = sortie de `scale(...)` / `scale_eggs(...)` / `scale_time(...)`
  (sous-linéaire pour le sel/les épices, discret pour les œufs, géométrique pour
  le temps, figé pour les températures).
- **écart** = `moteur − ×N naïf` (un écart non nul signale là où la cuisine réelle
  s'écarte de la règle de trois).

> ⚠️ **Disclaimer (cf. `table-scaling-sale.json` → `meta.disclaimer`)** : les
> coefficients sont des valeurs **de DÉPART** issues de la recherche + physique,
> **à CALIBRER** par des tests réels (×2 / ÷2). Ne pas les présenter comme exacts.

> 🧪 **Amorce (sprint 1).** Ce notebook tourne ici sur une **petite liste codée
> en dur** d'ingrédients représentatifs : il ne dépend PAS du corpus S0.6. La
> version *complète* (boucle sur les 10-15 recettes `recipes/` issues de S0.6,
> comparaison ×2 / ÷2 avec de la cuisine réelle) sera finalisée **après S0.6**
> (sprint 2) — voir la cellule de clôture.


In [ ]:
# Le moteur est pur/déterministe : aucun I/O réseau, aucun appel LLM ici.
# On importe depuis la racine du repo (lancer Jupyter depuis la racine, ou
# ajuster sys.path comme ci-dessous pour une exécution « Run All » robuste).
import sys
from pathlib import Path

# Rend le notebook ré-exécutable quel que soit le cwd : on remonte jusqu'à la
# racine du repo (dossier contenant `app/`).
_here = Path.cwd()
for _root in (_here, *_here.parents):
    if (_root / "app" / "scaling" / "engine.py").exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break

from app.scaling.engine import scale, scale_eggs, scale_time  # noqa: E402
from app.scaling.table import classify, load_table  # noqa: E402

_meta = load_table().get("meta", {})
print("Table :", _meta.get("name"), "v" + str(_meta.get("version")))
print("Disclaimer :", _meta.get("disclaimer"))


In [ ]:
# Panel d'ingrédients représentatifs (un par type de scaling), codé en dur pour
# l'amorce. (name, qty, unit) — la qté est arbitraire mais réaliste.
PANEL = [
    ("sel", 200, "g"),      # sublinear (coeff 0.75) -> réserve 10 %
    ("tomate", 200, "g"),   # linear (défaut)
    ("ail", 30, "g"),       # sublinear (aromate puissant)
    ("piment", 5, "g"),     # sublinear + flag non-linéarité
    ("oeuf", 3, "u"),       # discrete (œufs liants, RZ1)
    ("temps", 20, "min"),   # geometric (k^0.6667)
]


def moteur_value(name: str, qty: float, unit: str, k: float) -> float:
    """Valeur « moteur » pour un ingrédient du panel, en aiguillant vers la
    bonne fonction selon le type de scaling lu dans la table."""
    rule = classify(name)
    if rule.type == "discrete":          # œufs : on compare le NOMBRE d'œufs
        return float(scale_eggs(int(qty), k).whole_eggs)
    if rule.type == "geometric":         # temps : minutes scalées
        return scale_time(qty, k).minutes
    return scale(name, qty, unit, k).value  # linear / sublinear / fixed


def comparer(k: float) -> list[dict]:
    """Construit la table comparative qty_base | ×N_naïf | moteur | écart
    pour un facteur k donné, sans dépendance lourde (liste de dict)."""
    rows = []
    for name, qty, unit in PANEL:
        naif = qty * k
        moteur = moteur_value(name, qty, unit, k)
        rows.append(
            {
                "ingrédient": name,
                "type": classify(name).type,
                "qty_base": qty,
                "×N_naïf": round(naif, 3),
                "moteur": round(moteur, 3),
                "écart": round(moteur - naif, 3),
            }
        )
    return rows


def afficher(rows: list[dict]) -> None:
    """Rend la table en texte aligné (pas de pandas requis)."""
    cols = ["ingrédient", "type", "qty_base", "×N_naïf", "moteur", "écart"]
    widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in cols}
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    print(header)
    print("-" * len(header))
    for r in rows:
        print(" | ".join(str(r[c]).ljust(widths[c]) for c in cols))


In [ ]:
# ×2 : doublement. C'est le cas canonique. On attend un écart ~nul pour le
# linéaire (tomate) et NÉGATIF pour le sous-linéaire (sel, ail, piment) et le
# temps (le moteur scale MOINS que la règle de trois).
print("=== Facteur k = 2.0 (×2) ===")
afficher(comparer(2.0))


In [ ]:
# ÷2 et ×4 : on vérifie que la divergence est cohérente dans les deux sens.
for k in (0.5, 4.0):
    label = "÷2" if k == 0.5 else "×4"
    print(f"\n=== Facteur k = {k} ({label}) ===")
    afficher(comparer(k))


In [ ]:
# Lecture rapide (amorce) : l'écart est ~0 pour la tomate (linéaire == naïf),
# et s'éloigne de 0 pour sel/ail/piment/temps. À ×2, le sel passe de 400 (naïf)
# à ~336 (moteur) : c'est exactement la divergence que la calibration F2 doit
# valider sur de la cuisine réelle.
rows = comparer(2.0)
ecart_max = max(rows, key=lambda r: abs(r["écart"]))
print("Plus gros écart à ×2 :", ecart_max["ingrédient"], "->", ecart_max["écart"])
assert abs(next(r for r in rows if r["ingrédient"] == "tomate")["écart"]) < 1e-9, \
    "le linéaire doit égaler le ×N naïf"
print("OK — le linéaire égale le ×N naïf, le sous-linéaire/géométrique diverge.")


## Suite : version complète après S0.6

Cette amorce compare le moteur au `×N` naïf sur une **liste figée**
d'ingrédients. La version **complète** branchera le **corpus réel** :

- **S0.6** fournit 10-15 recettes italiennes (`recipes/`, format `cook.md`).
- On bouclera sur ces recettes pour comparer `moteur` vs `×N naïf` **par
  ingrédient réel**, aux facteurs **×2 / ÷2** (cuisine testée en vrai).
- Les écarts alimenteront la **boucle de calibration F2** (ajustement des
  coefficients de `docs/scaling/table-scaling-sale.json`) et le **jeu de
  validation E3**.

📎 Références : `docs/recherche-ouverte.md` §C (calibration) ;
`docs/architecture.md` §11 (ligne « Calibration scaling (moteur vs ×N naïf) »),
§13 R3 ; `table-scaling-sale.json` → `meta.disclaimer`.
